<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Computer-Networks/04-routing-and-control-plane.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Computer Networks guideline](Computer-Networks.html)


## **Routing and the Internet Control Plane**

Chapter 3 treated the forwarding table as an input: a router validated one packet, performed longest-prefix matching, chose a next hop, and transmitted through the data plane. This chapter asks where that forwarding state comes from. Routers must discover reachability, compare alternatives, distribute changes, reject loops, and translate administrative policy into entries that the fast path can use.

The running case remains an HTTPS packet for `203.0.113.0/24`. Inside a campus network, a link-state protocol finds a path from the access router to a border router. At the border, BGP compares customer, peer, and provider advertisements. Beyond the campus, independently operated networks make their own decisions, so the end-to-end path emerges from many local policies rather than one global optimizer.

::: {.callout-note}
On a first reading, prioritize the Bellman-Ford idea behind distance vector, the flood-then-Dijkstra idea behind link state, the difference between an IGP and BGP, and the roles of `AS_PATH`, `LOCAL_PREF`, and route export policy. Protocol timers, OSPF LSA types, route reflection, and RPKI add implementation and operational depth after that model is stable.
:::

Routing is a distributed-systems problem. Information arrives late, messages can be lost, links fail during calculation, and no router initially sees the whole Internet. A good design must eventually find useful paths while preventing transient disagreement from becoming persistent loops or widespread instability.


### **The Routing Problem**

#### **Topology, Metrics, Paths, and Policies**

A routing calculation commonly models a network as a directed graph $G=(V,E)$. A vertex represents a router or another routing object; an edge represents usable adjacency. A non-negative cost $c(u,v)$ describes the configured cost of forwarding from $u$ to $v$. Direction matters because capacity, policy, and even reachability can be asymmetric.

For an additive metric, the cost of path $p=(v_0,v_1,\ldots,v_k)$ is

$$
C(p)=\sum_{i=0}^{k-1}c(v_i,v_{i+1}).
$$

The least-cost path is the path with minimum $C(p)$ among eligible paths. The mathematics is simple only after the network has defined what an edge and a cost mean. A cost might approximate delay, inverse bandwidth, hop count, administrative preference, or a composite value. Some objectives are not additive: path capacity is constrained by the minimum-capacity edge, while commercial policy can reject a numerically short path entirely.

![The same topology selects different paths for latency, configured cost, capacity, or policy.](assets/routing-objectives.svg){fig-alt="Two paths between routers have different latency, capacity, configured cost, and policy preference" width="96%"}

The figure's upper route minimizes latency, while the lower route minimizes configured cost and offers more bottleneck capacity. A policy may prefer neither. There is therefore no universal "shortest" route independent of the metric and eligibility rules.

A route also needs more than a path cost. Practical state can include destination prefix, next hop, outgoing interface, source protocol, administrative preference, metric, age, route tag, and attributes used for advertisement. The control plane typically follows a lifecycle:

```text
receive or originate candidate reachability
    -> validate syntax, next hop, loop and policy conditions
    -> compare eligible candidates for the same prefix
    -> select one or more best paths
    -> resolve their next hops and install forwarding state
    -> advertise only what export policy permits
```

The selected control-plane route is not itself the forwarded packet. It programs the FIB described in Chapter 3, and the FIB may contain equal-cost next hops, a discard action, a summary, or a more-specific exception.

#### **Centralized vs Distributed Routing**

In a **distributed** protocol, routers exchange information and each computes local state. Distance-vector, OSPF, IS-IS, and BGP follow this broad model, although they distribute different information. No single participant must calculate every path.

A **logically centralized** controller collects topology or policy, calculates intent, and programs devices through control interfaces. Software-defined networking often uses this approach within one administrative domain. "Logically centralized" does not require one physical server; controllers can be replicated while exposing one coherent control service.

| Design | Knowledge during calculation | Strength | Main risk |
|---|---|---|---|
| Distributed | Partial or replicated through protocol messages | Autonomy and failure tolerance | Convergence and inconsistent intermediate views |
| Logically centralized | Controller assembles a wider view | Global optimization and uniform policy | Stale controller state, control reachability, scaling |
| Hybrid | Distributed base plus controller guidance | Practical balance | Interactions between two sources of intent |

The Internet is necessarily distributed across organizations. A campus or data centre can use stronger central coordination, but it still needs behavior for controller partitions, device restarts, and links that fail faster than new state can be installed.

#### **Static vs Dynamic Routes**

A **static route** is installed by configuration rather than learned from a routing protocol. Static routes are predictable and useful for defaults, stubs, discard summaries, management paths, and controlled failover. They do not automatically discover arbitrary topology changes.

A **dynamic route** is learned and withdrawn through a protocol. Dynamic routing adapts, but adaptation introduces messages, timers, authentication, algorithmic state, and transient behavior. A common design combines connected routes, selected static routes, an intradomain protocol, and BGP, then ranks candidates by route source before comparing each protocol's own metric.

The administrative preference between route sources is separate from the protocol metric. An OSPF cost of 10 and a BGP AS-path length of 3 are not values in one shared numeric scale. The router first decides which kinds of candidate are eligible and preferred, then applies protocol-specific comparison.

#### **Convergence, Stability, and Failure Recovery**

**Convergence** is the process by which participants incorporate a change and settle on mutually compatible forwarding state. Useful dimensions include:

- **detection time:** how quickly a node recognizes neighbor or link failure;
- **propagation time:** how quickly new information reaches affected routers;
- **calculation time:** how quickly routes are recomputed and selected;
- **installation time:** how quickly the FIB changes across devices;
- **traffic recovery time:** when packets again reach a valid destination.

During convergence, some routers use old state while others use new state. This can produce a temporary black hole, loop, path stretch, duplication, or reordering. Even if every router runs a correct algorithm, unsynchronized FIB installation can create a **microloop**. Fast failure detection is not automatically safe: reacting to every brief signal fluctuation can cause route flapping and repeated recalculation.

Routing therefore balances liveness and stability. It must react fast enough to real failure, but cautiously enough to avoid amplifying noise. Timers, dampening, ordered updates, loop-free alternates, graceful restart, and fast reroute address different parts of that trade-off.


In [1]:
paths = {
    "upper R1-R2-R4": {"latency_ms": 8, "admin_cost": 10, "bottleneck_gbps": 1},
    "lower R1-R3-R4": {"latency_ms": 15, "admin_cost": 4, "bottleneck_gbps": 10},
}

# Different routing objectives order exactly the same candidate paths differently.
lowest_latency = min(paths, key=lambda name: paths[name]["latency_ms"])
lowest_cost = min(paths, key=lambda name: paths[name]["admin_cost"])
highest_capacity = max(paths, key=lambda name: paths[name]["bottleneck_gbps"])

print("lowest latency: ", lowest_latency)
print("lowest cost:    ", lowest_cost)
print("highest capacity:", highest_capacity)


lowest latency:  upper R1-R2-R4
lowest cost:     lower R1-R3-R4
highest capacity: lower R1-R3-R4


### **Distance-Vector Routing**

#### **Bellman-Ford Recurrence**

In distance-vector routing, router $x$ maintains an estimated distance $D_x(y)$ to each destination $y$ and learns its neighbors' estimates. If $N(x)$ is the set of neighbors of $x$, the Bellman-Ford recurrence is

$$
D_x(y)=\min_{v\in N(x)}\left(c(x,v)+D_v(y)\right).
$$

The expression says: to reach $y$, choose a neighbor $v$, pay the local cost to $v$, then rely on $v$'s advertised distance to $y$. The minimizing neighbor becomes the next hop. A router does not initially need the full topology; it needs local link costs and vectors received from adjacent routers.

At initialization, a router knows distance zero to itself, direct cost to each neighbor, and infinity to other destinations. Every received vector can improve or invalidate entries. Repeated relaxation propagates knowledge farther: after one exchange, a router can learn useful two-hop paths; after more exchanges, information crosses a larger network.

![Distance-vector routers exchange their current destination costs with adjacent routers.](assets/distance-vector-overview.svg){fig-alt="Distance vector routing diagram with neighboring routers exchanging cost vectors" width="78%"}

*Figure source: [Webaware, DVA, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:DVA.svg), released into the public domain.*

The conceptual update is:

```text
ON_VECTOR_FROM(neighbor v, advertised_vector)
    for each destination y in advertised_vector
        candidate <- cost_to(v) + advertised_vector[y]
        compare candidate with current route to y
    if any selected route changed
        advertise the new vector according to protocol policy
```

Real implementations retain next hop, timers, route source, subnet mask, and changed-state flags rather than only one number. Updates may be periodic, triggered by change, or both. Asynchronous execution means neighbors need not process the same event in the same order.

#### **Distributed Route Advertisement**

Distance-vector messages reveal selected distances, not a complete graph. This reduces topology state at each router but creates dependence on neighbors' claims. Router A can know that B reports cost 3 to a prefix without knowing every link inside B's route.

The following executable model uses synchronous rounds only to make convergence visible. Infinity is a large teaching value, direct links are bidirectional, and every router advertises its complete vector each round. Real protocols optimize message format, timing, and changed entries.


In [2]:
from copy import deepcopy
from math import inf


graph = {
    "A": {"B": 1, "C": 4},
    "B": {"A": 1, "C": 1, "D": 7},
    "C": {"A": 4, "B": 1, "D": 2, "E": 6},
    "D": {"B": 7, "C": 2, "E": 1},
    "E": {"C": 6, "D": 1},
}


def distance_vector_rounds(topology: dict[str, dict[str, int]]):
    """Run synchronous Bellman-Ford rounds and retain each vector snapshot."""

    nodes = sorted(topology)
    vectors = {
        node: {
            destination: (
                0 if destination == node
                else topology[node].get(destination, inf)
            )
            for destination in nodes
        }
        for node in nodes
    }
    history = [deepcopy(vectors)]

    while True:
        updated = deepcopy(vectors)
        for node in nodes:
            for destination in nodes:
                if node == destination:
                    updated[node][destination] = 0
                    continue
                updated[node][destination] = min(
                    link_cost + vectors[neighbor][destination]
                    for neighbor, link_cost in topology[node].items()
                )
        history.append(updated)
        if updated == vectors:
            return history
        vectors = updated


history = distance_vector_rounds(graph)
print("round | A->D | A->E | B->E")
for round_number, vectors in enumerate(history):
    print(
        f"{round_number:5} | {vectors['A']['D']:4} | "
        f"{vectors['A']['E']:4} | {vectors['B']['E']:4}"
    )
print("converged rounds:", len(history) - 1)


round | A->D | A->E | B->E
    0 |  inf |  inf |  inf
    1 |    6 |   10 |    7
    2 |    4 |    7 |    4
    3 |    4 |    5 |    4
    4 |    4 |    5 |    4
converged rounds: 4


#### **Count-to-Infinity and Routing Loops**

Good news often propagates naturally: a newly shorter route is immediately attractive. Bad news is harder. If a destination disappears, two routers can each mistake the other's stale advertisement for an independent alternative.

![Routers A and B repeatedly increase a false distance after destination X becomes unreachable.](assets/dv-count-to-infinity.svg){fig-alt="Distance vector count to infinity timeline after a destination fails behind router C" width="98%"}

In the figure, B once reached X through C with cost 1 and A reached X through B with cost 2. After C loses X, B can hear A's stale claim of 2 and infer a route of 3 through A. A then hears B's 3 and infers 4. Packets loop between A and B while the metric increases. This is **count-to-infinity**: participants slowly discover that no finite route exists.

The problem is epistemic, not an arithmetic bug. A vector says "I can reach X at cost 2" but does not reveal whether that claimed route secretly comes back through the recipient. Path-vector protocols address this differently by carrying the sequence of autonomous systems.


In [3]:
def count_to_infinity(rounds: int = 8, protocol_infinity: int = 16):
    """Model A and B reinforcing stale routes after C loses destination X."""

    distance_a, distance_b = 2, 1
    timeline = [(0, distance_a, distance_b)]

    for round_number in range(1, rounds + 1):
        # Both routers calculate from the previous round's advertisement.
        new_a = min(protocol_infinity, 1 + distance_b)
        new_b = min(protocol_infinity, 1 + distance_a)
        distance_a, distance_b = new_a, new_b
        timeline.append((round_number, distance_a, distance_b))
    return timeline


print("round | A advertises | B advertises")
for round_number, distance_a, distance_b in count_to_infinity():
    print(f"{round_number:5} | {distance_a:12} | {distance_b:12}")


round | A advertises | B advertises
    0 |            2 |            1
    1 |            2 |            3
    2 |            4 |            3
    3 |            4 |            5
    4 |            6 |            5
    5 |            6 |            7
    6 |            8 |            7
    7 |            8 |            9
    8 |           10 |            9


#### **Split Horizon, Poison Reverse, and Hold-Down**

Distance-vector protocols combine several defenses:

- **Split horizon:** do not advertise a route back over the interface from which it was learned.
- **Poison reverse:** advertise that reverse route with infinity, explicitly telling the supplier not to use the receiver as an alternative.
- **Triggered updates:** send important changes without waiting for the next periodic update.
- **Route poisoning:** mark a failed route unreachable while retaining it long enough for the bad news to propagate.
- **Hold-down or invalidation timers:** avoid immediately accepting suspicious alternatives after failure.
- **Bounded infinity:** use a finite unreachable value so counting terminates.

Split horizon and poison reverse stop common two-router loops but not every loop involving several routers or complex redistribution. Hold-down improves stability at the cost of delaying legitimate recovery. The correct lesson is not that one timer "solves loops"; several mechanisms constrain different failure sequences.

RIP Version 2, defined by [RFC 2453](https://datatracker.ietf.org/doc/rfc2453/), uses hop count, valid metrics 1 through 15, and 16 as infinity. That small limit bounds count-to-infinity but also makes RIP unsuitable for paths requiring 16 or more hops.


### **Link-State Routing**

#### **Topology Discovery and Link-State Flooding**

Link-state routing distributes a different object. Each router describes its own local adjacencies and costs in a **Link-State Advertisement (LSA)** or protocol data unit. These records are reliably flooded through a defined scope, allowing routers in that scope to construct a consistent **Link-State Database (LSDB)**.

![Routers originate local link state, flood it reliably, build a shared database, and run shortest-path first independently.](assets/link-state-flooding.svg){fig-alt="Three phases of link state routing from neighbor discovery through flooding to Dijkstra shortest path calculation" width="98%"}

A link-state record needs identity and freshness, commonly including an originator, link descriptions, metric, sequence number, age or lifetime, and checksum. Only the originator should create authoritative new state for its own links. Other routers flood the record and reject stale duplicates.

The broad pipeline is:

1. Discover neighbors and establish required adjacencies.
2. Measure or configure local link costs.
3. Originate newer link-state information after change and periodically refresh it.
4. Reliably flood it through the area or level.
5. Build a graph from the newest accepted records.
6. Run shortest-path first with the local router as root.
7. derive next hops and install eligible prefixes in the FIB.

All routers can hold the same area graph but produce different shortest-path trees because each uses itself as root. The database represents topology facts; the tree and FIB represent one router's perspective.

#### **Dijkstra's Shortest-Path Algorithm**

Dijkstra's algorithm finds shortest paths from one source when edge costs are non-negative. It maintains tentative distances and repeatedly settles the unsettled vertex with smallest known distance.

```text
DIJKSTRA(graph, source)
    distance[source] <- 0; all other distances <- infinity
    candidate queue <- source
    while candidate queue is not empty
        u <- unsettled vertex with minimum distance
        settle u
        for each edge (u, v) with cost w
            if distance[u] + w < distance[v]
                distance[v] <- distance[u] + w
                parent[v] <- u
                update v in candidate queue
    return distances and predecessor tree
```

The update `distance[u] + w < distance[v]` is **relaxation**. Once the minimum candidate is removed from a correct priority queue, no later non-negative route can make it cheaper. With an adjacency list and binary heap, the usual complexity is $O((|V|+|E|)\log |V|)$; implementation details and graph density affect the practical cost.

![Animated execution of Dijkstra's algorithm on a weighted graph.](assets/dijkstra-animation.gif){fig-alt="Animation showing Dijkstra settling vertices and relaxing weighted edges" width="44%"}

*Figure source: [Ibmua, Dijkstra Animation, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Dijkstra_Animation.gif), released into the public domain.*

An IGP must define deterministic tie behavior and preserve equal-cost alternatives where ECMP is supported. The algorithm returns path costs, but route installation still needs prefix attachment, first-hop derivation, next-hop resolution, and protocol preference.


In [4]:
from heapq import heappop, heappush


def dijkstra(topology: dict[str, dict[str, int]], source: str):
    """Return distances, parents, and settlement order for a weighted graph."""

    distances = {node: inf for node in topology}
    parents = {node: None for node in topology}
    distances[source] = 0
    queue = [(0, source)]
    settled_order = []

    while queue:
        distance_u, node_u = heappop(queue)
        if distance_u != distances[node_u]:
            continue                     # Ignore an outdated heap entry.
        settled_order.append((node_u, distance_u))

        for node_v, edge_cost in topology[node_u].items():
            candidate = distance_u + edge_cost
            if candidate < distances[node_v]:
                distances[node_v] = candidate
                parents[node_v] = node_u
                heappush(queue, (candidate, node_v))
    return distances, parents, settled_order


def reconstruct_path(parents: dict[str, str | None], destination: str):
    path = []
    current = destination
    while current is not None:
        path.append(current)
        current = parents[current]
    return list(reversed(path))


distances, parents, settled = dijkstra(graph, "A")
print("settlement order:", settled)
for destination in sorted(graph):
    print(
        f"A -> {destination}: cost={distances[destination]:2}, "
        f"path={'-'.join(reconstruct_path(parents, destination))}"
    )


settlement order: [('A', 0), ('B', 1), ('C', 2), ('D', 4), ('E', 5)]
A -> A: cost= 0, path=A
A -> B: cost= 1, path=A-B
A -> C: cost= 2, path=A-B-C
A -> D: cost= 4, path=A-B-C-D
A -> E: cost= 5, path=A-B-C-D-E


#### **Sequence Numbers, Aging, and Reliable Flooding**

Flooding must distinguish a new description from a delayed duplicate. A larger sequence number normally identifies a newer instance from the same originator. Age or lifetime eventually removes records whose originator has disappeared. Checksums detect damaged LSA content, while acknowledgments and retransmission improve reliable distribution.

These mechanisms interact with restart. A router that reboots cannot safely originate a sequence number that every neighbor considers older than the pre-restart record. Protocols define initialization, maximum-sequence, flush, and refresh behavior to recover a coherent database.

Reliable flooding does not mean every router changes its FIB simultaneously. One receives the LSA first, another receives it later, and SPF scheduling can batch changes. Operators tune detection, flooding, SPF throttling, and installation to limit CPU storms while meeting recovery goals.

#### **Distance Vector vs Link State**

| Property | Distance vector | Link state |
|---|---|---|
| Advertised knowledge | Selected distance/reachability | Local links and attributes |
| Router's topology view | Usually incomplete | Replicated within flooding scope |
| Core computation | Bellman-Ford relaxation | Dijkstra SPF |
| Loop information | Next hop and metric may hide dependency | Full scoped topology enables explicit paths |
| Failure behavior | Bad news can count or loop | Flood new state, then recompute; microloops remain possible |
| State and processing | Vectors per destination | LSDB plus SPF structures |
| Typical examples | RIP | OSPF and IS-IS |

Neither family is universally "better." Scope, scale, topology, convergence goals, implementation quality, and operational tooling matter. BGP is a path-vector protocol: it shares some incremental, neighbor-to-neighbor behavior with distance vector, but carries an AS path and rich policy attributes rather than one additive distance.


### **Intradomain Routing**

An **Interior Gateway Protocol (IGP)** operates within one administrative routing domain. The organization controls metrics, addressing, failure policy, authentication, and hierarchy, so it can pursue consistent internal objectives. IGP reachability is also used to resolve BGP next hops: BGP may select an external route, while OSPF or IS-IS determines how an internal router reaches the chosen border.

#### **Routing Information Protocol**

RIP is a distance-vector IGP whose metric is hop count. A directly reachable route normally has metric 1; 16 is unreachable. RIPv2 adds classless masks, route tags, next-hop information, multicast updates, and authentication support compared with classful RIP behavior.

Its small diameter and timer-driven convergence make RIP useful for teaching and limited simple networks, not for a large modern backbone. A path with two slow links can appear better than one fast link because hop count ignores bandwidth and delay. The protocol demonstrates why metric semantics matter as much as algorithm correctness.

#### **Open Shortest Path First**

OSPFv2 is specified by [RFC 2328](https://datatracker.ietf.org/doc/rfc2328/). Its operation connects several state machines:

1. **Hello processing** discovers neighbors, checks compatible parameters, and detects liveness.
2. **Adjacency formation** decides which neighbors exchange complete database state. On multi-access networks, a Designated Router and Backup Designated Router reduce adjacency and flooding complexity.
3. **Database synchronization** exchanges summaries and requests missing or newer LSAs.
4. **Reliable flooding** distributes LSAs within their defined scope.
5. **SPF calculation** builds a shortest-path tree from the LSDB.
6. **Route calculation and installation** derives intra-area, inter-area, and external routes plus next hops.

OSPF cost is additive and administratively derived, often from reference bandwidth divided by interface bandwidth. Defaults can make several high-speed links appear equal unless the reference is updated. Cost is not measured application latency and does not automatically react to transient queueing.

OSPF uses areas to bound LSDB size and recalculation scope. **Area 0** is the backbone for inter-area distribution. An Area Border Router connects areas; an Autonomous System Boundary Router redistributes external reachability. Summarization at boundaries reduces state and hides churn, but a summary must not attract traffic that cannot be delivered. Equal-cost paths can produce ECMP next hops.

OSPF route types and LSA scopes matter when debugging. Two routers can agree on physical links yet choose differently because one route is intra-area and another is external, because area configuration is inconsistent, or because redistribution changed type and metric semantics.

#### **IS-IS and Operational Differences**

IS-IS is also a link-state IGP and runs SPF over a flooded database. Integrated IS-IS carries IP reachability using extensible Type-Length-Value fields. Unlike OSPF's IP protocol transport, IS-IS control packets run directly over the link layer, so an IP-addressing failure does not necessarily prevent adjacency formation.

Its hierarchy uses Level 1 for routing within an area and Level 2 between areas; a router can participate in both. OSPF and IS-IS differ in packet formats, terminology, area attachment, flooding details, and operational culture, but both implement the central pattern of neighbor discovery, topology flooding, SPF, hierarchy, and route installation.

| Concern | OSPF | IS-IS |
|---|---|---|
| Control transport | Directly over IP | Directly over link layer |
| Hierarchy | Area 0 backbone plus other areas | Level 1 within area, Level 2 between areas |
| Extensibility | LSA and protocol extensions | Strong TLV-based extensibility |
| IPv4/IPv6 deployment | OSPFv2 for IPv4, OSPFv3 commonly for IPv6 | Multi-protocol reachability in integrated IS-IS |

#### **Areas, Summarization, ECMP, and Redistribution**

Hierarchy limits the blast radius of detailed topology changes. Summaries reduce table size and update frequency; ECMP uses multiple equal-cost next hops, normally hashing a flow to avoid reordering. These are not free optimizations. Summaries can hide failure, ECMP can expose flows to unequal path quality, and hierarchy can create suboptimal paths.

**Redistribution** imports routes from another protocol or source. It is a frequent source of loops because the imported route can be exported back without its original meaning. Safe design uses explicit direction, tags or communities, metric mapping, filtering, and a clear ownership boundary. "Redistribute everything both ways" makes two independently correct protocols form one ambiguous feedback system.


### **Autonomous Systems and Internet Economics**

An **Autonomous System (AS)** is a set of routed networks under coordinated administration that presents routing policy to other networks. It is identified in BGP by an Autonomous System Number (ASN). An AS is an administrative and policy object, not necessarily one company, one building, or one IGP area.

#### **Transit, Peering, and Customer-Provider Relationships**

Interdomain routing reflects business relationships because carrying third-party traffic consumes capacity and has economic value.

![Representative customer, provider, peering, and Internet exchange relationships among autonomous systems.](assets/inter-as-peering-transit.svg){fig-alt="Autonomous systems connect through customer provider transit and peer relationships at Internet exchanges" width="84%"}

*Figure source: [Bill Woodcock, Inter-AS peering and transit relationships, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Inter-AS_peering_and_transit_relationships_01.svg), licensed under CC BY-SA 4.0.*

- A **customer** pays a **provider** for transit to destinations the provider can reach. A provider usually advertises broad reachability to its customer.
- **Peers** exchange traffic for their own and their customers' destinations, commonly without agreeing to carry traffic between unrelated providers.
- An **Internet Exchange Point (IXP)** supplies shared switching infrastructure where networks establish bilateral sessions or use a route server to simplify multilateral exchange. The IXP itself is not automatically the transit provider for all exchanged traffic.

Commercial terms are more nuanced than three labels: paid peering, partial transit, regional constraints, traffic ratios, capacity commitments, and private interconnects exist. The simple model is still valuable because it explains common import preference and export restrictions.

#### **Policy Routing vs Shortest-Path Routing**

An AS commonly prefers customer-learned routes over peer routes and peer routes over provider routes, because customer traffic produces revenue while provider traffic incurs cost. This local preference can dominate AS-path length. The globally observed route is therefore not guaranteed to minimize hops, latency, or geography.

Within an AS, **hot-potato routing** often sends traffic to the nearest acceptable egress according to the IGP, reducing the AS's own internal carriage. A different egress could have better end-to-end latency. BGP chooses an external policy path; the IGP cost to the BGP next hop can then choose where the traffic exits.

#### **Valley-Free Paths and Export Policies**

A simplified economically consistent path goes "up" zero or more customer-to-provider links, optionally crosses one peer link, then goes "down" zero or more provider-to-customer links. It does not go down to a customer and then back up through that customer's provider, because the middle AS would be giving free transit between providers or peers.

Typical export behavior is:

| Route learned from | Export to customers | Export to peers | Export to providers |
|---|---:|---:|---:|
| Own prefix | Yes | Yes | Yes |
| Customer | Yes | Yes | Yes |
| Peer | Yes | No | No |
| Provider | Yes | No | No |

This table expresses a common model, not a protocol-enforced law. A route leak occurs when advertisement violates intended propagation policy. Because BGP neighbors cannot infer private business contracts from packet fields alone, prevention requires filters, communities, relationship signaling, monitoring, and coordination.


In [5]:
def is_valley_free(relationships: list[str]) -> bool:
    """Check the simplified c2p* p2p? p2c* relationship pattern."""

    phase = "up"
    for relationship in relationships:
        if phase == "up":
            if relationship == "c2p":
                continue
            if relationship == "p2p":
                phase = "peer"
                continue
            if relationship == "p2c":
                phase = "down"
                continue
        elif phase == "peer":
            if relationship == "p2c":
                phase = "down"
                continue
        elif phase == "down" and relationship == "p2c":
            continue
        return False
    return True


examples = {
    "customer -> provider -> peer -> customer": ["c2p", "p2p", "p2c"],
    "provider -> customer -> provider": ["p2c", "c2p"],
    "customer -> provider -> provider -> customer": ["c2p", "c2p", "p2c"],
    "peer -> peer": ["p2p", "p2p"],
}

for description, relationships in examples.items():
    print(f"{description:46} -> {is_valley_free(relationships)}")


customer -> provider -> peer -> customer       -> True
provider -> customer -> provider               -> False
customer -> provider -> provider -> customer   -> True
peer -> peer                                   -> False


### **Border Gateway Protocol**

BGP-4, specified by [RFC 4271](https://datatracker.ietf.org/doc/rfc4271/), is the Internet's inter-AS routing protocol. It is a **path-vector** protocol: an advertisement associates a prefix with an `AS_PATH` and other attributes. The path supports loop rejection and policy, while incremental UPDATE messages announce and withdraw reachability.

#### **eBGP and iBGP Sessions**

**eBGP** exchanges routes between different autonomous systems. **iBGP** distributes BGP routes among speakers inside one AS. Both normally use long-lived TCP sessions on port 179. The TCP connection carries control messages; user packets do not travel "inside the BGP session."

A BGP speaker passes through a session state machine and exchanges:

- **OPEN** to negotiate identity and capabilities;
- **KEEPALIVE** to confirm an established session remains active;
- **UPDATE** to announce reachable prefixes with path attributes or withdraw them;
- **NOTIFICATION** to report an error and close the session;
- **ROUTE-REFRESH** capability and related mechanisms to request readvertisement without resetting the session.

BGP is incremental. Once peers have exchanged their initial routing state, they send changes rather than periodically retransmitting the complete Internet table. TCP provides ordered reliable delivery between peers, but it does not make the advertised routes truthful or globally consistent.

#### **BGP Route Advertisements and Path Attributes**

An UPDATE can carry Network Layer Reachability Information (NLRI), withdrawn routes, and attributes. Important attributes include:

| Attribute | Meaning in selection or propagation |
|---|---|
| `AS_PATH` | AS sequence traversed; rejects a route containing the local ASN and often prefers shorter paths after local policy |
| `NEXT_HOP` | IP next hop that must be reachable through the local routing system |
| `LOCAL_PREF` | Preference used inside one AS; higher is commonly preferred and it is not sent to external peers |
| `MED` | Suggestion to a neighboring AS about preferred entry; lower is commonly preferred among comparable routes |
| `ORIGIN` | How the route entered BGP: IGP, EGP, or incomplete |
| `COMMUNITIES` | Policy tags used to group routes and control preference or export |

Communities, defined initially by [RFC 1997](https://datatracker.ietf.org/doc/rfc1997/), carry policy labels rather than physical topology. An operator might tag customer routes, request that a provider not export a route in one region, or mark a route for blackholing under an agreed convention.

`AS_PATH` is not a cryptographic proof of every hop. It is an attribute propagated and modified according to BGP rules. Origin validation and emerging path-validation mechanisms address different portions of trust.

#### **Route Selection and Policy**

BGP first rejects infeasible or policy-denied candidates, then applies a decision process. A useful conceptual order is:

1. highest local preference or locally computed degree of preference;
2. shortest `AS_PATH` among routes still tied;
3. preferred `ORIGIN` type;
4. lowest MED where the routes are meaningfully comparable;
5. eBGP-learned over iBGP-learned in a tie;
6. lowest IGP cost to the BGP next hop;
7. deterministic implementation-specific tie-breakers.

Exact ordering and additional stages vary by implementation and configuration. Vendor-specific values such as weight should not be mistaken for a universal BGP attribute.

![Three routes to one prefix are compared; a longer customer path wins because LOCAL_PREF is evaluated before AS_PATH length.](assets/bgp-policy-selection.svg){fig-alt="BGP route selection prefers customer local preference 200 over shorter peer and provider AS paths" width="98%"}

Only routes tied at an earlier stage reach a later stage. A provider path with one AS hop does not beat a customer path with three AS hops when local preference already selected the customer route.


In [6]:
from dataclasses import dataclass, replace


@dataclass(frozen=True)
class BGPRoute:
    learned_from: str
    local_pref: int
    as_path: tuple[int, ...]
    origin: str
    ebgp: bool
    igp_cost_to_next_hop: int
    router_id: str


ORIGIN_RANK = {"IGP": 0, "EGP": 1, "INCOMPLETE": 2}


def bgp_selection_key(route: BGPRoute):
    """A transparent teaching subset of a common BGP decision process."""

    return (
        -route.local_pref,                 # Higher LOCAL_PREF wins.
        len(route.as_path),                # Then prefer the shorter AS path.
        ORIGIN_RANK[route.origin],         # IGP before EGP before incomplete.
        0 if route.ebgp else 1,            # Prefer eBGP in this simplified tie.
        route.igp_cost_to_next_hop,        # Hot-potato exit among remaining ties.
        route.router_id,                   # Deterministic final tie-break.
    )


candidates = [
    BGPRoute("customer", 200, (65100, 65210, 65300), "IGP", True, 30, "10.0.0.3"),
    BGPRoute("peer", 150, (65400, 65300), "IGP", True, 10, "10.0.0.2"),
    BGPRoute("provider", 100, (65500,), "IGP", True, 5, "10.0.0.1"),
]

best_policy_route = min(candidates, key=bgp_selection_key)
equal_local_pref = [replace(route, local_pref=100) for route in candidates]
best_short_path = min(equal_local_pref, key=bgp_selection_key)

print("normal policy winner:   ", best_policy_route.learned_from,
      best_policy_route.as_path)
print("if LOCAL_PREF is equal: ", best_short_path.learned_from,
      best_short_path.as_path)


normal policy winner:    customer (65100, 65210, 65300)
if LOCAL_PREF is equal:  provider (65500,)


#### **Import Policy, Export Policy, and the FIB**

**Import policy** decides which received routes are accepted and how attributes such as local preference are set. The decision process selects a best route for local use. **Export policy** then decides whether and how that route is announced to each neighbor. Receiving, selecting, installing, and advertising are separate events.

The selected BGP route must have a resolvable next hop. An internal router may use OSPF or IS-IS to reach an egress router named by `NEXT_HOP`. This recursion connects the control planes:

```text
BGP selects external prefix and egress
    -> IGP supplies path to BGP next hop
    -> adjacency resolves the immediate link neighbor
    -> FIB installs prefix and one or more forwarding actions
```

Changing the IGP cost can move traffic between already-acceptable BGP exits without changing the external AS path. Conversely, a BGP withdrawal can remove the external route while internal connectivity remains healthy.

#### **Route Reflection and Scaling iBGP**

Under the traditional rule, $n$ iBGP speakers need

$$
\frac{n(n-1)}{2}
$$

sessions for a full mesh. That grows quadratically. Route reflection, standardized by [RFC 4456](https://datatracker.ietf.org/doc/rfc4456/), lets a **Route Reflector (RR)** advertise selected iBGP routes to clients that do not peer with one another.

![A BGP route reflector distributes routes among client and non-client iBGP speakers.](assets/bgp-route-reflection.svg){fig-alt="BGP route reflection topology with route reflector clients and external peers" width="70%"}

*Figure source: [Shiyu Ji, RR BGP, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:RR_BGP.svg), licensed under CC BY-SA 4.0.*

`ORIGINATOR_ID` and `CLUSTER_LIST` help prevent reflection loops. Route reflection reduces session count but can hide alternative paths because an RR normally advertises only its selected best path. Placement and policy can therefore produce suboptimal routing, inconsistent visibility, or control-plane paths that do not align with forwarding topology. Redundant reflectors, add-path mechanisms, and careful design address different scaling and path-diversity needs.


### **Convergence and Interdomain Failure Modes**

#### **Path Exploration and Slow Convergence**

When a BGP route is withdrawn, a router may try several previously advertised alternatives, announce one, later learn that it is invalid, and try another. This **path exploration** spreads through ASes and can take much longer than one link propagation delay. Policy, update ordering, session timers, route-reflector visibility, and rate controls all influence convergence.

A route flap repeatedly announces and withdraws reachability. Dampening can suppress unstable routes but may also delay recovery of a route that became stable. Modern operation therefore combines selective dampening, session protection, prefix limits, fast failure detection where appropriate, and monitoring rather than applying one aggressive timer everywhere.

#### **Route Leaks and Prefix Hijacking**

A **prefix hijack** occurs when an AS originates or propagates unauthorized reachability that attracts traffic for another network's prefix. A **route leak**, classified in [RFC 7908](https://datatracker.ietf.org/doc/html/rfc7908), propagates a learned route beyond its intended scope, often violating customer-provider or peer export policy. A leak can involve the legitimate origin AS and still create an unintended transit path.

More-specific prefixes normally win longest-prefix matching, so an unauthorized `/24` can attract traffic away from a legitimate covering `/20` even when the `/20` has attractive BGP attributes. LPM occurs in the data plane after control-plane selection; BGP preference does not override a different prefix length.

Consequences include blackholing, interception, congestion, asymmetric paths, and widespread instability. Some incidents are malicious, but configuration errors and incorrect redistribution are sufficient.

#### **Filtering, RPKI, and Route Origin Validation**

The **Resource Public Key Infrastructure (RPKI)** binds IP address and ASN resources to signed objects. A Route Origin Authorization (ROA) states that an AS may originate a prefix up to a maximum length. A relying-party validator verifies repositories and supplies validated ROA payloads to routers.

![A ROA produces Valid, Invalid, and Not Found route-origin-validation states for different BGP announcements.](assets/rpki-origin-validation.svg){fig-alt="RPKI route origin validation compares prefix maximum length and origin AS to classify BGP routes" width="98%"}

As described by [RFC 6811](https://datatracker.ietf.org/doc/html/rfc6811), an announcement is conceptually:

- **Valid** when a covering ROA permits both its origin AS and prefix length;
- **Invalid** when a covering ROA exists but the origin or allowed length does not match;
- **Not Found** when no validated ROA covers it.

The validation state is an input to local policy; RPKI does not directly change BGP packets on the wire. Common policy rejects Invalid routes while treating Not Found according to deployment strategy. Incorrect ROAs can invalidate legitimate announcements, so certificate and ROA operations are part of production routing reliability.

ROV validates the origin, not every AS in `AS_PATH`, the commercial relationship between neighbors, or end-to-end service identity. It helps with mis-origination but does not by itself stop all route leaks or path manipulation.

Other controls remain necessary:

- prefix and maximum-length filters based on customer authorization;
- maximum-prefix limits to contain accidental table floods;
- bogon and unallocated-space filtering;
- AS-path and community policy;
- source-address validation at network edges;
- monitoring of unexpected origins, path changes, and reachability;
- protected routing sessions and disciplined configuration change.

#### **Operational Tradeoffs in Routing Security**

Strict filters reduce attack and error propagation but can discard legitimate emergency announcements if authorization data is stale. Permissive policy preserves reachability but spreads mistakes. Partial deployment means protected networks still interact with unvalidated paths. Safe rollout needs observability, staged enforcement, fail-safe validator architecture, accurate resource records, and a process for correcting false invalids.


In [7]:
from ipaddress import ip_network


@dataclass(frozen=True)
class ROA:
    prefix: object
    max_length: int
    origin_as: int


def origin_validation(announced_prefix: str, origin_as: int, roas: list[ROA]):
    """Return the RFC 6811-style Valid, Invalid, or Not Found state."""

    announcement = ip_network(announced_prefix)
    covering = [roa for roa in roas if announcement.subnet_of(roa.prefix)]
    if not covering:
        return "Not Found"
    if any(
        origin_as == roa.origin_as
        and announcement.prefixlen <= roa.max_length
        for roa in covering
    ):
        return "Valid"
    return "Invalid"


validated_roas = [ROA(ip_network("203.0.113.0/24"), 24, 65001)]
announcements = [
    ("203.0.113.0/24", 65001),  # Authorized exact prefix.
    ("203.0.113.0/24", 65009),  # Wrong origin.
    ("203.0.113.0/25", 65001),  # Too specific for maxLength 24.
    ("198.51.100.0/24", 65008), # No covering ROA.
]

for prefix, origin in announcements:
    print(f"{prefix:18} origin AS{origin} -> "
          f"{origin_validation(prefix, origin, validated_roas)}")


203.0.113.0/24     origin AS65001 -> Valid
203.0.113.0/24     origin AS65009 -> Invalid
203.0.113.0/25     origin AS65001 -> Invalid
198.51.100.0/24    origin AS65008 -> Not Found


### **Anycast, Multicast, and Specialized Routing**

#### **IP Anycast and Service Replication**

With **anycast**, several service sites originate reachability to the same service prefix. Ordinary routing selects one instance according to topology and policy. The address is the same; the chosen physical server or site depends on the client-side routing view.

Anycast is used for DNS infrastructure, content systems, attack absorption, and other replicated services. It can localize traffic and provide coarse failover when an unhealthy site withdraws its route. However, "nearest" means best according to routing policy, not necessarily lowest geographic distance or measured latency.

[RFC 4786](https://datatracker.ietf.org/doc/rfc4786/) emphasizes routing stability relative to transaction duration. If a route changes during a stateful session, later packets can reach a different site that lacks the connection state. Designs use stable routing, short transactions, state replication, application retries, or connection-aware architecture according to service requirements.

Anycast is also not uniform load balancing. One site may attract much more traffic because AS policies and prefix propagation are uneven. Health signaling must withdraw only when the service, not merely the router interface, is unusable.

#### **Multicast Trees and Group Membership Overview**

Unicast sends one packet toward one destination address. Multicast sends traffic to a group, and routers replicate packets only where delivery paths branch. This can avoid sending many identical unicast copies over the same upstream link.

Hosts signal local group interest with IGMP for IPv4 or MLD for IPv6. Routing protocols such as PIM build distribution trees. Source-Specific Multicast uses an `(S,G)` channel for one source and group; Any-Source Multicast can use shared-tree and rendezvous-point mechanisms before switching toward a source-specific path.

Multicast introduces per-group state, receiver membership changes, reverse-path forwarding checks, tree construction, and boundary policy. It is effective in controlled domains for live media, market data, and replication, but global deployment is operationally more complex than ordinary unicast.

| Delivery model | Address meaning | Selection | Main operational concern |
|---|---|---|---|
| Unicast | One endpoint | Route to that destination | Path quality and failover |
| Anycast | One replicated service address | Routing chooses one instance | Stability and uneven attraction |
| Multicast | Receiver group | Tree reaches joined receivers | Group state, replication, and scope |


### **Simulating Distance-Vector and Link-State Routing**

The earlier code cells used one topology for two views. Distance vector propagated destination costs over several rounds. Link state assumed that flooding had produced a graph, then Dijkstra calculated all paths locally. A failure changes both systems, but through different control sequences:

```text
Distance vector:
detect -> change local vector -> advertise -> neighbors relax -> repeat until stable

Link state:
detect -> originate newer LSA -> flood -> each router reruns SPF -> install
```

The comparison below removes the low-cost B-C link. It does not simulate packet timing or asynchronous messages; it verifies the route that a correctly converged link-state database should produce before and after the event.


In [8]:
def remove_undirected_link(topology, node_a: str, node_b: str):
    changed = deepcopy(topology)
    del changed[node_a][node_b]
    del changed[node_b][node_a]
    return changed


failed_graph = remove_undirected_link(graph, "B", "C")

before_distances, before_parents, _ = dijkstra(graph, "A")
after_distances, after_parents, _ = dijkstra(failed_graph, "A")

print("before B-C failure:",
      reconstruct_path(before_parents, "E"), "cost", before_distances["E"])
print("after B-C failure: ",
      reconstruct_path(after_parents, "E"), "cost", after_distances["E"])

# A reasonableness check: the failed edge must not appear in the new path.
new_edges = set(zip(
    reconstruct_path(after_parents, "E"),
    reconstruct_path(after_parents, "E")[1:],
))
assert ("B", "C") not in new_edges and ("C", "B") not in new_edges


before B-C failure: ['A', 'B', 'C', 'D', 'E'] cost 5
after B-C failure:  ['A', 'C', 'D', 'E'] cost 7


The new path is more expensive but valid. A production convergence test would additionally model asynchronous detection, message loss, protocol timers, SPF scheduling, FIB installation, and packets in flight. Algorithmic convergence and traffic recovery are related measurements, not synonyms.

For a network with $|V|$ routers and $|E|$ links, a heap-based SPF calculation is commonly described as $O((|V|+|E|)\log |V|)$. Flooding one changed LSA requires transmissions across the flooding topology. A naive distance-vector round sends destination vectors over neighbor adjacencies and may require many rounds. These asymptotic descriptions help compare mechanisms but do not predict convergence time without protocol timing, hardware, topology, and failure details.

### **Tracing an Internet Route Across Autonomous Systems**

Return to the HTTPS packet for `203.0.113.0/24`:

1. The host's default route sends the packet to the campus gateway.
2. The gateway's FIB contains a route derived from the campus IGP toward a border router.
3. The border router has several BGP candidates and selects one according to local policy.
4. Its `NEXT_HOP` is resolved through the IGP and installed as a forwarding action.
5. The provider AS selects and exports its own BGP path; later ASes repeat the process.
6. The destination AS, or an anycast service site originating the same prefix, delivers the packet internally.

The visible IP-hop trace and BGP AS path are different evidence:

| Evidence | Shows | Does not directly show |
|---|---|---|
| `tracert` / traceroute | Interfaces returning TTL-expired responses and RTT samples | Every hidden hop, complete router identity, or exact AS policy |
| Local route table | Selected local prefix, next hop, interface, metric/source | Remote AS decisions |
| BGP looking glass | Routes visible to one BGP speaker | The exact path used by every client |
| Route collector | Historical or current control-plane advertisements | Guaranteed data-plane forwarding |
| RPKI validator | Route-origin authorization state | Full AS-path validity or service ownership |

Mapping traceroute addresses to ASNs is imperfect. Router interfaces can use addresses from another organization, IXPs provide shared subnets, MPLS can hide hops, and the return path for ICMP can differ. Per-flow load balancing can also show different interfaces when probe headers vary.

On Windows, useful local commands include:

```powershell
Get-NetRoute -AddressFamily IPv4
route print
tracert 203.0.113.8
Test-NetConnection 203.0.113.8 -Port 443
```

Use documentation addresses only for examples; they are not expected to provide a real service. For a live investigation, record time, source network, destination, address family, probe method, and route-collector viewpoint. Routing changes while evidence is being collected.

### **Comparison and Summary**

| Do not confuse | First concept | Second concept |
|---|---|---|
| Forwarding vs routing | Applies installed state to each packet | Learns, selects, and distributes paths |
| Metric vs policy | Numeric comparison inside an objective | Eligibility and preference rules that can dominate metrics |
| Distance vector vs link state | Exchanges selected distances | Floods local topology state and runs SPF |
| IGP vs BGP | Optimizes reachability inside one administration | Exchanges policy reachability among autonomous systems |
| eBGP vs iBGP | Connects different ASes | Distributes BGP routes within one AS |
| `AS_PATH` vs physical hop list | Control-plane sequence used for policy and loop rejection | Every router or interface traversed by packets |
| Route leak vs hijack | Exports legitimate learned reachability beyond intended scope | Attracts traffic through unauthorized origin/propagation |
| RPKI ROV vs full path security | Validates prefix length and origin authorization | Does not validate every AS relationship or application endpoint |
| Anycast vs multicast | Routing chooses one replicated service instance | A tree delivers to multiple joined receivers |

Inside one AS, RIP illustrates Bellman-Ford distance exchange, while OSPF and IS-IS flood topology and compute shortest-path trees. Between ASes, BGP carries paths and attributes so each organization can apply economic, security, and engineering policy. Convergence is never just an algorithm finishing: detection, propagation, calculation, installation, and traffic behavior all matter.

The routing system has now selected a path and the IP data plane can forward packets along it. Chapter 5 moves to the endpoints and asks how TCP, UDP, and QUIC identify applications, detect loss, order data, manage flow control, and build reliable end-to-end communication over those changing best-effort paths.
